In [1]:
import numpy as np

#Funciones a usar

def sol_trsupcol(A, b):
    n = len(b)
    x = b.copy()

    for idx in reversed(range(n)):
        if b[idx] != 0:
            j = idx
            break

    for i in reversed(range(j + 1)):
        x[i] = x[i] / A[i, i]
        x[:i] = x[:i] - A[:i, i] * x[i]

    return x


def givens(x1,x2):
    c = 1.
    s = 0.
    ax1 = abs(x1)
    ax2 = abs(x2)
    if ax1 + ax2 > 0:
        if ax2 > ax1:
            tau = -x1/x2
            s = -np.sign(x2)/np.sqrt(1 + tau**2)
            c = tau*s
        else:
            tau = -x2/x1
            c = np.sign(x1)/np.sqrt(1 + tau**2)
            s = tau*c
    return c, s


def qrgivensp(A):
    m, n = A.shape
    Q = np.eye(m)
    P = np.eye(n)
    R = A.copy()
    # Forma rápida de calcular la norma de todas las columnas de A
    c_norm = np.linalg.norm(A, axis=0) ** 2
    p = min(m - 1, n)

    for j in range(p):
        # Pivoteo por derecha, cambiando columnas en vez de filas
        ind_pivot = j + np.argmax(c_norm[j:])
        if ind_pivot != j:
            R[:, [ind_pivot, j]] = R[:, [j, ind_pivot]]
            P[:, [ind_pivot, j]] = P[:, [j, ind_pivot]]
            c_norm[[ind_pivot, j]] = c_norm[[j, ind_pivot]]
        # Givens común y corriente
        for i in range(j+1, m):
            if R[i, j] != 0:
                #I = [j, i], J = j:
                c, s = givens(R[j, j], R[i, j])
                G = np.array([[c, -s], [s, c]])

                R[[j, i], j:] = G @ R[[j, i], j:]
                Q[:, [j, i]] = Q[:, [j, i]] @ G.T
        # Actualización de c_norm SOLO UNA VEZ DESPUES DE ACTUALIZAR FILAS
        c_norm[j:] = c_norm[j:] - R[j, j:] * R[j, j:]

    if m <= n and R[m - 1, m - 1] < 0:
        # J = m:
        R[m - 1, m - 1:] = -R[m - 1, m - 1:]
        Q[:, m - 1] = -Q[:, m - 1]

    return Q, R, P

######################################################

#EJERCICIO 7

def sol_cuadmin(A, b):
    m, n = A.shape
    Q, R, P = qrgivensp(A)

    y = np.zeros(n)
    q = Q.T @ b
    r = np.diag(R)
    p = min(m,n)
    busqueda_ceros = np.where(r == 0)[0] #np.where nos devuelve el indice o la posicion del elemnto que estamos buscando
    if len(busqueda_ceros) != 0:
        p = busqueda_ceros[0] #Aqui es mejor que nos devuelva el primer elemento que es cero en la diagonal de R

    y[:p] = sol_trsupcol(R[:p, :p], q[:p])

    x = P@y

    return  x, np.linalg.norm(q[p:])

#TEST para sol_cuadmin
A = np.random.random((2, 3))
b = np.random.random(2)
x, res_2 = sol_cuadmin(A, b)
print(A@x-b)

NameError: name 'qrgivensp' is not defined